# Model Comparison: OutcomePE, TimescalePE, VectorRPE

Visualises the predictions of four model types in two tasks:
- **Parker task** (RPE mode): left/right lever reversal bandit
- **Jin & Costa task** (APE mode): sequential 3-press lever task

Models:
| Key | Class | Signal |
|-----|-------|--------|
| `vrpe` | `VectorRPEAgent` | Feature-specific RPE (Lee et al. 2024) |
| `outcome` | `OutcomePEAgent` | Distributional / Expectile RPE (Dabney et al. 2020) |
| `tpe_flat` | `TimescalePEAgent` β=1 | Custom β model, uniform weights |
| `tpe_bias` / `tpe_ts` | `TimescalePEAgent` β≠1 | Custom β model, reward/timescale bias |

Run `parkerTPE.py` and `jincostaTPE.py` first to generate the data.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm

matplotlib.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
})

COLORS = {
    'vrpe':       '#2c7bb6',
    'outcome':    '#d7191c',
    'tpe_flat':   '#1a9641',
    'tpe_bias':   '#fdae61',
    'tpe_ts':     '#f46d43',
    'reward':     '#d62728',
    'omission':   '#9467bd',
    'left':       '#2ca02c',
    'right':      '#ff7f0e',
}

---
## 1  Parker Task (RPE mode)

In [ ]:
p = np.load('./data/parker/TPE/DA.npz', allow_pickle=True)

da_vrpe      = p['da_vrpe']       # (N_trials,) object array; each element = list of (num_features,) arrays
da_outcome   = p['da_outcome']    # each element = list of (num_channels,) arrays
da_tpe_flat  = p['da_tpe_flat']   # each element = list of (num_features,) arrays
da_tpe_bias  = p['da_tpe_bias']   # each element = list of (num_features,) arrays

actions       = p['actions']
rewards       = p['rewards']
reward_trials = p['reward_trials'].tolist()
states_log    = p['states']
epoch         = p['epoch']
taus          = p['taus_outcome']
betas_flat    = p['betas_flat']
betas_biased  = p['betas_biased']

num_trials         = len(da_vrpe)
num_features       = da_vrpe[0][0].shape[0]
num_outcome_ch     = da_outcome[0][0].shape[0]

# State labels
state_labels = ['start', 'reward', 'omit',
                'left-1', 'left-2', 'left-3',
                'right-1', 'right-2', 'right-3']
LEFT_FEAT_IDX  = 3   # first left-path feature (parkerRPE convention)
RIGHT_FEAT_IDX = 6   # first right-path feature

print(f'Loaded {num_trials} trials | features={num_features} | outcome_channels={num_outcome_ch}')
print(f'Reward trials: {len(reward_trials)}')
print(f'Outcome channel taus: {np.round(taus, 3)}')
print(f'Biased betas: {np.round(betas_biased, 3)}')

In [ ]:
def find_choice_idx(trial_states):
    """Index of first non-start state (= action commitment moment)."""
    for i, s in enumerate(trial_states):
        if s != 0:
            return i
    return None


def get_event_responses(da_array, feature_idx, reward_trials, states_log, num_trials):
    """
    Extract mean PE at 4 task events for a given feature/channel index:
      0 = start, 1 = choice, 2 = post-choice, 3 = outcome

    Returns dict with keys 'reward' and 'omission', each containing
    arrays of shape (4,) = [start, choice, post-choice, outcome].
    """
    omit_trials = [i for i in range(num_trials) if i not in reward_trials]

    def _resp(trial_list):
        acc = np.zeros(4)
        n   = 0
        for idx in trial_list:
            trial_da     = da_array[idx]
            trial_states = states_log[idx]
            ci = find_choice_idx(trial_states)
            if ci is None or ci + 1 >= len(trial_da):
                continue
            acc[0] += trial_da[0][feature_idx]
            acc[1] += trial_da[ci][feature_idx]
            acc[2] += trial_da[min(ci + 1, len(trial_da)-1)][feature_idx]
            acc[3] += trial_da[-1][feature_idx]
            n += 1
        return acc / max(n, 1)

    return {'reward': _resp(reward_trials), 'omission': _resp(omit_trials)}


def get_scalar_event_responses(da_array, reward_trials, states_log, num_trials):
    """
    Same but for scalar (or already-selected) DA values.
    da_array[trial][step] should be a scalar or indexable by [0].
    """
    omit_trials = [i for i in range(num_trials) if i not in reward_trials]

    def _resp(trial_list):
        acc = np.zeros(4); n = 0
        for idx in trial_list:
            trial_da     = da_array[idx]
            trial_states = states_log[idx]
            ci = find_choice_idx(trial_states)
            if ci is None or ci + 1 >= len(trial_da):
                continue
            acc[0] += float(trial_da[0])
            acc[1] += float(trial_da[ci])
            acc[2] += float(trial_da[min(ci+1, len(trial_da)-1)])
            acc[3] += float(trial_da[-1])
            n += 1
        return acc / max(n, 1)

    return {'reward': _resp(reward_trials), 'omission': _resp(omit_trials)}


EVENTS = ['Start', 'Choice', 'Post-choice', 'Outcome']
print('Helper functions defined.')

### 1.1  VectorRPE vs OutcomePE at choice & outcome
Replicates the logic of Figure 8d but for RPE models.  
Left-path feature (index 3) and right-path feature (index 6) are shown.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5), sharey='row')
fig.suptitle('VectorRPE (top) vs OutcomePE (bottom) — Parker task', fontsize=12)

xlabels  = EVENTS
bar_kw   = dict(width=0.35, capsize=3)
x        = np.arange(len(EVENTS))

# ── Row 0: VectorRPE — left-tuned and right-tuned features ──
for col, (feat_idx, feat_name) in enumerate([
        (LEFT_FEAT_IDX,  'Left-path feature'),
        (RIGHT_FEAT_IDX, 'Right-path feature')]):
    resp = get_event_responses(da_vrpe, feat_idx, reward_trials, states_log, num_trials)
    ax   = axes[0, col]
    ax.bar(x - 0.18, resp['reward'],   color=COLORS['reward'],   label='Reward',   **bar_kw)
    ax.bar(x + 0.18, resp['omission'], color=COLORS['omission'], label='Omission', **bar_kw)
    ax.set_title(f'VectorRPE\n{feat_name}')
    ax.set_xticks(x); ax.set_xticklabels(xlabels, rotation=20, ha='right')
    ax.axhline(0, color='k', lw=0.7)
    if col == 0:
        ax.set_ylabel('Mean δ')
    ax.legend(fontsize=8)

# Hide unused top panels
for col in [2, 3]:
    axes[0, col].set_visible(False)

# ── Row 1: OutcomePE — most pessimistic and most optimistic channel ──
for col, (ch_idx, ch_name, color) in enumerate([
        (0,                   f'Pessimistic  τ={taus[0]:.2f}',  '#d62728'),
        (num_outcome_ch // 2, f'Median        τ={taus[num_outcome_ch//2]:.2f}', '#ff7f0e'),
        (num_outcome_ch - 2,  f'Optimistic    τ={taus[-2]:.2f}',  '#2ca02c'),
        (num_outcome_ch - 1,  f'Most optimistic τ={taus[-1]:.2f}', '#1f77b4'),
    ]):
    resp = get_event_responses(da_outcome, ch_idx, reward_trials, states_log, num_trials)
    ax   = axes[1, col]
    ax.bar(x - 0.18, resp['reward'],   color=COLORS['reward'],   label='Reward',   **bar_kw)
    ax.bar(x + 0.18, resp['omission'], color=COLORS['omission'], label='Omission', **bar_kw)
    ax.set_title(f'OutcomePE\n{ch_name}')
    ax.set_xticks(x); ax.set_xticklabels(xlabels, rotation=20, ha='right')
    ax.axhline(0, color='k', lw=0.7)
    if col == 0:
        ax.set_ylabel('Mean δ')
    if col == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('./data/parker/TPE/fig1_vrpe_vs_outcome.pdf', bbox_inches='tight')
plt.show()

### 1.2  OutcomePE: reversal-point spectrum across all channels
At reward outcome, optimistic channels (high τ) respond more strongly than pessimistic ones.  
At omission, the gradient reverses.

In [ ]:
outcome_resps_reward   = []
outcome_resps_omission = []

omit_trials = [i for i in range(num_trials) if i not in reward_trials]

for ch in range(num_outcome_ch):
    resp = get_event_responses(da_outcome, ch, reward_trials, states_log, num_trials)
    outcome_resps_reward.append(resp['reward'][3])    # index 3 = outcome timestep
    outcome_resps_omission.append(resp['omission'][3])

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
fig.suptitle('OutcomePE: distribution of reversal points across τ channels', fontsize=11)

norm  = Normalize(vmin=0, vmax=1)
cmap  = cm.RdYlGn
colors_tau = [cmap(norm(t)) for t in taus]

ax = axes[0]
for i, (rew, tau) in enumerate(zip(outcome_resps_reward, taus)):
    ax.bar(i, rew, color=colors_tau[i])
ax.set_xticks(range(num_outcome_ch))
ax.set_xticklabels([f'{t:.2f}' for t in taus], rotation=45)
ax.set_xlabel('Expectile τ  (pessimistic → optimistic)')
ax.set_ylabel('Mean δ at outcome')
ax.set_title('Reward trials')
ax.axhline(0, color='k', lw=0.7)

ax = axes[1]
for i, (om, tau) in enumerate(zip(outcome_resps_omission, taus)):
    ax.bar(i, om, color=colors_tau[i])
ax.set_xticks(range(num_outcome_ch))
ax.set_xticklabels([f'{t:.2f}' for t in taus], rotation=45)
ax.set_xlabel('Expectile τ')
ax.set_title('Omission trials')
ax.axhline(0, color='k', lw=0.7)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=axes, label='τ (pessimistic → optimistic)', shrink=0.7)
plt.tight_layout()
plt.savefig('./data/parker/TPE/fig2_outcome_reversal.pdf', bbox_inches='tight')
plt.show()

### 1.3  TimescalePE: flat β vs reward-biased β
The biased β model weights left-path features more heavily (β>1) and right-path less (β<1),
reflecting the initial reward probability (left=0.7, right=0.1).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharey='row')
fig.suptitle('TimescalePE: flat β (top) vs reward-biased β (bottom)', fontsize=12)

pairs = [
    (LEFT_FEAT_IDX,  'Left-path feat',  da_tpe_flat,  da_tpe_bias),
    (RIGHT_FEAT_IDX, 'Right-path feat', da_tpe_flat,  da_tpe_bias),
    (1,              'Reward-state feat', da_tpe_flat, da_tpe_bias),
]

for col, (fi, name, da_flat, da_bias) in enumerate(pairs):
    for row, (da_arr, label, betas) in enumerate([
            (da_flat, 'Flat β', betas_flat),
            (da_bias, 'Biased β', betas_biased)]):
        resp = get_event_responses(da_arr, fi, reward_trials, states_log, num_trials)
        ax   = axes[row, col]
        x_ev = np.arange(len(EVENTS))
        ax.bar(x_ev - 0.18, resp['reward'],   color=COLORS['reward'],   width=0.35, label='Reward')
        ax.bar(x_ev + 0.18, resp['omission'], color=COLORS['omission'], width=0.35, label='Omission')
        ax.set_xticks(x_ev)
        ax.set_xticklabels(EVENTS, rotation=20, ha='right')
        ax.axhline(0, color='k', lw=0.7)
        ax.set_title(f'{label} — {name}\nβ={betas[fi]:.2f}')
        if col == 0:
            ax.set_ylabel('Mean δ')
        if col == 0 and row == 0:
            ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('./data/parker/TPE/fig3_tpe_flat_vs_biased.pdf', bbox_inches='tight')
plt.show()

### 1.4  β-weight spectrum: per-feature β at outcome time
Shows how the β vector shapes the per-feature PE magnitude at reward vs omission.

In [ ]:
# Collect outcome-time PE for every feature, for biased model
feat_outcome_rew  = []
feat_outcome_omit = []

for fi in range(num_features):
    resp = get_event_responses(da_tpe_bias, fi, reward_trials, states_log, num_trials)
    feat_outcome_rew.append(resp['reward'][3])
    feat_outcome_omit.append(resp['omission'][3])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
fig.suptitle('TimescalePE biased β — per-feature PE at outcome', fontsize=11)

ax = axes[0]
ax.bar(range(num_features), betas_biased, color='#636363')
ax.set_xlabel('Feature index'); ax.set_ylabel('β_i')
ax.set_title('β weights')
ax.set_xticks(range(num_features))
ax.set_xticklabels(state_labels, rotation=45, ha='right', fontsize=8)

ax = axes[1]
ax.bar(range(num_features), feat_outcome_rew, color=COLORS['reward'])
ax.set_xlabel('Feature index'); ax.set_ylabel('Mean δ at outcome')
ax.set_title('Reward outcome: δ_i = r/(β_i N) + …')
ax.set_xticks(range(num_features))
ax.set_xticklabels(state_labels, rotation=45, ha='right', fontsize=8)
ax.axhline(0, color='k', lw=0.7)

ax = axes[2]
ax.bar(range(num_features), feat_outcome_omit, color=COLORS['omission'])
ax.set_xlabel('Feature index'); ax.set_ylabel('Mean δ at outcome')
ax.set_title('Omission outcome')
ax.set_xticks(range(num_features))
ax.set_xticklabels(state_labels, rotation=45, ha='right', fontsize=8)
ax.axhline(0, color='k', lw=0.7)

plt.tight_layout()
plt.savefig('./data/parker/TPE/fig4_beta_spectrum.pdf', bbox_inches='tight')
plt.show()

### 1.5  All four models side-by-side at choice and outcome
Summary figure: the 'signature channel' of each model at the two most informative events.

In [ ]:
# TimescaleRPE da signal is (N_VALUES,) per step — average across channels for scalar summary
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
fig.suptitle('Parker task: model comparison at choice and outcome', fontsize=11)

# Signature channels:
#   VectorRPE  → feature index LEFT_FEAT_IDX (left-path state tuning)
#   OutcomePE  → most optimistic channel (τ closest to 1)
#   TimescalePE flat/biased → channel 0 (left value channel), index into (N_VALUES,) vector
sig_chans = {
    'VectorRPE\n(left feat)':     (da_vrpe,     LEFT_FEAT_IDX),
    'OutcomePE\n(optimistic τ)':  (da_outcome,  num_outcome_ch - 1),
    'TimescalePE\n(flat β, left ch)':   (da_tpe_flat, 0),
    'TimescalePE\n(biased β, left ch)': (da_tpe_bias, 0),
}

for ax_idx, (event_idx, event_name) in enumerate([(1, 'Choice'), (3, 'Outcome')]):
    ax   = axes[ax_idx]
    rew_vals  = []
    omit_vals = []
    for (da_arr, fi) in sig_chans.values():
        resp = get_event_responses(da_arr, fi, reward_trials, states_log, num_trials)
        rew_vals.append(resp['reward'][event_idx])
        omit_vals.append(resp['omission'][event_idx])

    x_m = np.arange(len(sig_chans))
    ax.bar(x_m - 0.2, rew_vals,  width=0.38, color=COLORS['reward'],   alpha=0.85, label='Reward')
    ax.bar(x_m + 0.2, omit_vals, width=0.38, color=COLORS['omission'], alpha=0.85, label='Omission')
    ax.set_xticks(x_m)
    ax.set_xticklabels(list(sig_chans.keys()), fontsize=8)
    ax.axhline(0, color='k', lw=0.7)
    ax.set_title(event_name); ax.set_ylabel('Mean δ')
    if ax_idx == 0: ax.legend()

plt.tight_layout()
plt.savefig('./data/parker/TPE/fig5_all_models_summary.pdf', bbox_inches='tight')
plt.show()

---
## 2  Jin & Costa Task (APE mode)

In [ ]:
jc = np.load('./data/jincosta/TPE/DA.npz', allow_pickle=True)

da_jc = {
    'vrpe':     (jc['da_vrpe_L'],     jc['da_vrpe_R']),
    'outcome':  (jc['da_outcome_L'],  jc['da_outcome_R']),
    'tpe_flat': (jc['da_tpe_flat_L'], jc['da_tpe_flat_R']),
    'tpe_ts':   (jc['da_tpe_ts_L'],   jc['da_tpe_ts_R']),
}

jc_states   = jc['states']
jc_actions  = jc['actions']
taus_jc     = jc['taus_outcome']
betas_ts    = jc['betas_ts']
press_thresh= int(jc['press_thresh'])
n_jc_trials = len(jc_states)
jc_features = int(jc['chunked_features'])   # 21
jc_nch      = int(jc['num_outcome_channels'])

print(f'Loaded {n_jc_trials} trials | features={jc_features} | outcome_ch={jc_nch}')
print(f'β_ts (first 7): {np.round(betas_ts[:7], 3)}')

In [ ]:
def build_press_sequence_matrix(da_left, da_right, num_trials, press_thresh,
                                 feature_or_channel_idx):
    """
    Build a (num_trials × press_thresh+2) matrix of mean DA signal at
    key moments: [start, press1, press2, press3, outcome].
    Returns separate matrices for left-agent and right-agent.
    """
    n_steps = press_thresh + 2   # start + presses + outcome
    mat_L = np.full((num_trials, n_steps), np.nan)
    mat_R = np.full((num_trials, n_steps), np.nan)

    for t in range(num_trials):
        try:
            daL_t  = da_left[t]
            daR_t  = da_right[t]
            st_t   = jc_states[t]
            act_t  = jc_actions[t]

            fi = feature_or_channel_idx

            # Start = first step
            mat_L[t, 0] = float(np.asarray(daL_t[0]).flat[fi])
            mat_R[t, 0] = float(np.asarray(daR_t[0]).flat[fi])

            # Find press events: states where action is -1 → still ITI
            # actions list has one entry per press event (-1 = wait, 0/1 = press)
            press_steps = []
            for step_idx, action in enumerate(act_t):
                if action is not None and action != -1:
                    press_steps.append(step_idx + 1)  # +1 to align with da index

            for p, ps in enumerate(press_steps[:press_thresh]):
                if ps < len(daL_t):
                    mat_L[t, p + 1] = float(np.asarray(daL_t[ps]).flat[fi])
                    mat_R[t, p + 1] = float(np.asarray(daR_t[ps]).flat[fi])

            # Outcome = last step
            mat_L[t, -1] = float(np.asarray(daL_t[-1]).flat[fi])
            mat_R[t, -1] = float(np.asarray(daR_t[-1]).flat[fi])
        except Exception:
            pass

    return mat_L, mat_R


JC_XLABELS = ['Start'] + [f'Press {i+1}' for i in range(press_thresh)] + ['Outcome']
print('Jin & Costa helper functions defined.')

### 2.1  Heatmaps: DA signal across neurons × time steps
Each row = one feature channel; colour = mean DA at that step.  
Neurons sorted by feature index (= peak-tuned press number).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Jin & Costa task: DA signal heatmaps  (left agent, APE mode)', fontsize=12)

model_names = ['VectorRPE', 'OutcomePE', 'TimescalePE flat β', 'TimescalePE timescale β']
model_keys  = ['vrpe', 'outcome', 'tpe_flat', 'tpe_ts']
n_steps_jc  = press_thresh + 2

for col, (key, mname) in enumerate(zip(model_keys, model_names)):
    daL, daR = da_jc[key]

    if key == 'outcome':
        n_units = jc_nch
    else:
        n_units = jc_features

    # Build mean-response matrix: (n_units × n_steps_jc)
    mat = np.zeros((n_units, n_steps_jc))
    for fi in range(n_units):
        mL, _ = build_press_sequence_matrix(daL, daR, n_jc_trials, press_thresh, fi)
        col_mean = np.nanmean(mL, axis=0)  # mean over trials
        mat[fi, :] = col_mean

    vmax = np.nanpercentile(np.abs(mat), 97)

    for row, (mat_data, subtitle) in enumerate([(mat, 'Left agent')]):
        ax = axes[row, col]
        im = ax.imshow(mat_data, aspect='auto', cmap='RdBu_r',
                       vmin=-vmax, vmax=vmax, interpolation='nearest')
        ax.set_xticks(range(n_steps_jc))
        ax.set_xticklabels(JC_XLABELS, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Unit index')
        ax.set_title(f'{mname}\n({n_units} units)')
        plt.colorbar(im, ax=ax, shrink=0.7, label='Mean δ')

# Bottom row: timescale β visualised
for col in range(4):
    axes[1, col].set_visible(False)

# Add β heatmap for timescale model in bottom-right
ax = fig.add_subplot(2, 4, 8)
ax.bar(range(len(betas_ts)), betas_ts, color='#636363', width=1.0)
ax.set_xlabel('Feature index')
ax.set_ylabel('β_i')
ax.set_title('Timescale β weights\n(copy 1→0.5, copy 2→1.0, copy 3→1.5)')
ax.axvline(7,  color='orange', lw=1.5, ls='--', label='Copy boundary')
ax.axvline(14, color='orange', lw=1.5, ls='--')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('./data/jincosta/TPE/fig6_jc_heatmaps.pdf', bbox_inches='tight')
plt.show()

### 2.2  Mean time-course: averaged DA across all units vs time
Converges to the scalar RPE (VectorRPE) for feature-specific models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Jin & Costa: average DA signal across all units', fontsize=11)

for ax_idx, (agent_side, side_name) in enumerate([('L', 'Left agent'), ('R', 'Right agent')]):
    ax = axes[ax_idx]
    ax.set_title(side_name)

    for key, mname, color in [
        ('vrpe',     'VectorRPE',          COLORS['vrpe']),
        ('outcome',  'OutcomePE',           COLORS['outcome']),
        ('tpe_flat', 'TimescalePE flat β',  COLORS['tpe_flat']),
        ('tpe_ts',   'TimescalePE ts β',    COLORS['tpe_ts']),
    ]:
        daL, daR = da_jc[key]
        da_side  = daL if agent_side == 'L' else daR

        if key == 'outcome':
            n_units = jc_nch
        else:
            n_units = jc_features

        # Average over units then over trials
        step_means = np.zeros(press_thresh + 2)
        for fi in range(n_units):
            mL, _ = build_press_sequence_matrix(daL, daR, n_jc_trials, press_thresh, fi)
            step_means += np.nanmean(mL, axis=0)
        step_means /= n_units

        ax.plot(range(press_thresh + 2), step_means, '-o', color=color,
                label=mname, lw=2, ms=6)

    ax.set_xticks(range(press_thresh + 2))
    ax.set_xticklabels(JC_XLABELS, rotation=20, ha='right')
    ax.axhline(0, color='k', lw=0.7)
    ax.set_ylabel('Mean δ (avg over units)')
    if ax_idx == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('./data/jincosta/TPE/fig7_jc_timecourse.pdf', bbox_inches='tight')
plt.show()

### 2.3  OutcomePE channels: heterogeneity across τ in APE mode
In APE mode the 'reward' is an action indicator. Channels with high τ are 'optimistic'
about action frequency — they expect the action to occur often.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
fig.suptitle('OutcomePE (APE mode) — channel responses across τ', fontsize=11)

norm  = Normalize(vmin=0, vmax=1)
cmap  = cm.RdYlGn
colors_tau = [cmap(norm(t)) for t in taus_jc]

daL_out, daR_out = da_jc['outcome']

for ax_idx, (da_side, side_name) in enumerate([(daL_out, 'Left agent'), (daR_out, 'Right agent')]):
    ax = axes[ax_idx]
    ch_press1 = []   # PE at press-1 step
    ch_outcome= []   # PE at outcome step

    for ch in range(jc_nch):
        mat, _ = build_press_sequence_matrix(da_side, da_side, n_jc_trials, press_thresh, ch)
        mean_steps = np.nanmean(mat, axis=0)
        ch_press1.append(mean_steps[1])   # press 1 column
        ch_outcome.append(mean_steps[-1]) # outcome column

    x = np.arange(jc_nch)
    width = 0.35
    for i in range(jc_nch):
        ax.bar(i - 0.18, ch_press1[i],  width=width, color=colors_tau[i], alpha=0.9)
        ax.bar(i + 0.18, ch_outcome[i], width=width, color=colors_tau[i], alpha=0.5, hatch='//')

    # Legend proxies
    ax.bar([], [], color='gray', label='Press 1 step')
    ax.bar([], [], color='gray', alpha=0.5, hatch='//', label='Outcome step')
    ax.legend(fontsize=8)

    ax.set_xticks(range(jc_nch))
    ax.set_xticklabels([f'τ={t:.2f}' for t in taus_jc], rotation=45, fontsize=8)
    ax.axhline(0, color='k', lw=0.7)
    ax.set_ylabel('Mean δ')
    ax.set_title(side_name)

plt.tight_layout()
plt.savefig('./data/jincosta/TPE/fig8_jc_outcome_tau.pdf', bbox_inches='tight')
plt.show()

### 2.4  TimescalePE: timescale β vs flat β — per-copy feature responses
The timescale β model assigns higher weight to features in copy 3 (later presses),
creating a temporal gradient in the PE signal magnitude.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('TimescalePE — effect of timescale β on per-copy responses (Jin & Costa)', fontsize=11)

# Summarise each copy's mean absolute PE across press steps
copy_size   = jc_features // 3   # 7
copy_labels = ['Copy 1\n(β≈0.5)', 'Copy 2\n(β≈1.0)', 'Copy 3\n(β≈1.5)']

for col, (key, label, color) in enumerate([
        ('tpe_flat', 'Flat β (all=1)',      COLORS['tpe_flat']),
        ('tpe_ts',   'Timescale β',          COLORS['tpe_ts'])]):
    if col > 1: break
    daL, _ = da_jc[key]

    copy_responses = []
    for copy_idx in range(3):
        feat_start = copy_idx * copy_size
        feat_end   = feat_start + copy_size
        resp_per_copy = []
        for fi in range(feat_start, feat_end):
            mL, _ = build_press_sequence_matrix(daL, daL, n_jc_trials, press_thresh, fi)
            # mean abs PE across all press steps (excluding start)
            resp_per_copy.append(np.nanmean(np.abs(mL[:, 1:-1])))
        copy_responses.append(np.mean(resp_per_copy))

    ax = axes[col]
    bars = ax.bar(range(3), copy_responses, color=color, alpha=0.85)
    ax.set_xticks(range(3))
    ax.set_xticklabels(copy_labels)
    ax.set_ylabel('Mean |δ| across press steps')
    ax.set_title(label)
    for bar, val in zip(bars, copy_responses):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', fontsize=9)

# Panel 3: difference
daL_flat, _ = da_jc['tpe_flat']
daL_ts,   _ = da_jc['tpe_ts']
diffs = []
for copy_idx in range(3):
    feat_start = copy_idx * copy_size
    feat_end   = feat_start + copy_size
    diff_copy  = []
    for fi in range(feat_start, feat_end):
        mL_flat, _ = build_press_sequence_matrix(daL_flat, daL_flat, n_jc_trials, press_thresh, fi)
        mL_ts,   _ = build_press_sequence_matrix(daL_ts,   daL_ts,   n_jc_trials, press_thresh, fi)
        diff_copy.append(np.nanmean(np.abs(mL_ts[:, 1:-1])) -
                         np.nanmean(np.abs(mL_flat[:, 1:-1])))
    diffs.append(np.mean(diff_copy))

ax = axes[2]
bar_colors = ['#d62728' if d < 0 else '#2ca02c' for d in diffs]
ax.bar(range(3), diffs, color=bar_colors, alpha=0.85)
ax.set_xticks(range(3))
ax.set_xticklabels(copy_labels)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Δ|δ|  (timescale − flat)')
ax.set_title('Timescale effect\n(positive = amplified by β)')

plt.tight_layout()
plt.savefig('./data/jincosta/TPE/fig9_jc_beta_timescale.pdf', bbox_inches='tight')
plt.show()

---
## Summary of key predictions

| Model | Cue/Choice response | Outcome response | Heterogeneity axis |
|-------|--------------------|--------------------|--------------------|
| **VectorRPE** | Heterogeneous (feature-driven) | Uniform ($r_t/N$ shared) | State feature $\phi_i$ |
| **OutcomePE** | Mostly uniform (same features) | **Heterogeneous** (τ-sorted) | Pessimism/optimism τ |
| **TimescalePE flat** | Heterogeneous (= VectorRPE) | Uniform | State feature (β=1) |
| **TimescalePE biased/ts** | Heterogeneous + β-scaled | Graded ($r/(β_i N)$) | Feature × β weight |

### 1.6  β dynamics: learned channel gains over training
With `lr_beta = 2 × lr`, β adapts twice as fast as w.  
**Flat-start** shows β emerging from a uniform prior;  
**Biased-start** shows β refining a reward-probability prior.

In [ ]:
p2 = np.load('./data/parker/TPE/DA.npz', allow_pickle=True)

beta_flat_hist   = p2['beta_flat_history']     # (N_snapshots, N_VALUES)
beta_biased_hist = p2['beta_biased_history']   # (N_snapshots, N_VALUES)
snap_trials      = p2['beta_snapshot_trials']  # episode indices
N_VALUES_TPE     = beta_flat_hist.shape[1]     # = 2 (left / right)
chan_labels       = ['Left channel', 'Right channel']

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle(f'TimescaleRPE β dynamics (N_VALUES={N_VALUES_TPE}, α_β = 2×α_w)', fontsize=12)

chan_colors = ['#2ca02c', '#ff7f0e']

# ── Top row: β_i trajectory over trials ──
for row, (hist, title) in enumerate([
        (beta_flat_hist,   'Flat β start (all β=1)'),
        (beta_biased_hist, 'Biased β start (left=1.4, right=0.6)')]):
    ax = axes[row, 0]
    for ch in range(N_VALUES_TPE):
        ax.plot(snap_trials, hist[:, ch], color=chan_colors[ch],
                lw=1.8, label=chan_labels[ch])
    for bk in range(1, 4):
        ax.axvline(bk * int(p2['block_switch_interval']),
                   color='k', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(1.0, color='gray', lw=0.7, ls=':', label='β=1')
    ax.set_xlabel('Trial'); ax.set_ylabel('β_i')
    ax.set_title(title); ax.legend(fontsize=8)

# ── Bottom row: initial vs final β bar chart ──
for row, (hist, title) in enumerate([
        (beta_flat_hist,   'Flat start: initial → final β'),
        (beta_biased_hist, 'Biased start: initial → final β')]):
    ax   = axes[row, 1]
    init = hist[0]
    final= hist[-1]
    x    = np.arange(N_VALUES_TPE)
    ax.bar(x - 0.2, init,  width=0.38, color='#aec7e8', label='Initial β')
    ax.bar(x + 0.2, final, width=0.38,
           color=chan_colors, label='Final β')
    ax.set_xticks(x)
    ax.set_xticklabels(chan_labels, fontsize=9)
    ax.axhline(1.0, color='gray', lw=0.7, ls=':')
    ax.set_ylabel('β_i'); ax.set_title(title); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('./data/parker/TPE/fig_beta_dynamics.pdf', bbox_inches='tight')
plt.show()